# 31 - Export the four Figure 2 pages

Figure 2 shows four handwritten FERMAT pages. FERMAT is gated, so the pages are
not committed to this repository; this notebook is the one step that needs
Hugging Face access. **No GPU, no model, no inference.**

Each page is resolved by matching `sha256(orig_q)` **and** `sha256(pert_a)`
against `reference/wacv_evaluation_artifact/fermat_n300_public_manifest.csv`.
Matching on the question alone is not safe here: only 246 of the 300 questions
are distinct, and an earlier attempt to identify a Figure 2 panel that way
returned two candidates. The manifest holds hashes only, no gated text.

Writes `item_229.jpg`, `item_92.jpg`, `item_160.jpg`, `item_251.jpg` to Drive
first and then into the repo clone, and rebuilds the figure so you can see the
result here.

In [ ]:
# No model and no GPU: this notebook needs dataset access and a repo clone.
!pip install -q datasets huggingface_hub

In [ ]:
import json
import os
from getpass import getpass

from huggingface_hub import login
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("Stored HF token does not start with 'hf_'; set RESET_TOKENS = True.")
login(token=HF_TOKEN)
print("Hugging Face login OK")

# Clone into a commit-named directory so a stale checkout can never be reused,
# which is the failure notebook 30 hit four times.
REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf nb31_repo && git clone -q {REPO_URL} nb31_repo
%pip install -q -e nb31_repo/

import importlib
import sys

REPO_DIR = os.path.abspath("nb31_repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
for _n in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_n]

!git -C nb31_repo log --oneline -1

In [ ]:
# Resolve each page by hash pair. The manifest carries only hashes, so nothing
# gated is read out of the repository to do this.
import hashlib

import pandas as pd
import pilot.data

TARGETS = [229, 92, 160, 251]

man = pd.read_csv(f"{REPO_DIR}/reference/wacv_evaluation_artifact/"
                  "fermat_n300_public_manifest.csv").set_index("item_id")


def sha(text):
    return hashlib.sha256(str(text if text is not None else "").encode("utf-8")).hexdigest()


fermat = pilot.data.load_fermat_balanced(n=300, seed=42, target_error_frac=0.5)
print(f"FERMAT n=300 balanced: {len(fermat)} items")

by_pair = {}
for item in fermat:
    by_pair.setdefault((sha(item["orig_q"]), sha(item["pert_a"])), []).append(item)

resolved = {}
for item_id in TARGETS:
    key = (man.loc[item_id, "question_sha256"], man.loc[item_id, "reference_answer_sha256"])
    hits = by_pair.get(key, [])
    # A missing or ambiguous hit means the Hub copy drifted. Fail loudly rather
    # than guessing: this project has already had `shuffle(seed=42)` move two
    # items between draws that were meant to be disjoint.
    assert len(hits) == 1, (
        f"item {item_id}: expected exactly 1 hash match, got {len(hits)}. "
        "The dataset revision has drifted; do not guess a page.")
    resolved[item_id] = hits[0]
    print(f"  item {item_id:>3}  resolved by hash pair  has_error={hits[0]['has_error']}")

print(f"\nresolved {len(resolved)}/{len(TARGETS)}")

In [ ]:
# Drive FIRST, then the repo clone. Notebook 15 attached five images correctly
# and lost them all when the push 403'd and the ephemeral clone vanished; Drive
# is the source of truth here for exactly that reason.
import shutil
from pathlib import Path

DRIVE_PAGES = Path(PROJECT_DIR) / "figure2_pages"
REPO_PAGES = Path(REPO_DIR) / "paper" / "figures" / "pages"
DRIVE_PAGES.mkdir(parents=True, exist_ok=True)
REPO_PAGES.mkdir(parents=True, exist_ok=True)

for item_id, item in resolved.items():
    name = f"item_{item_id}.jpg"
    img = item["image"]
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.save(DRIVE_PAGES / name, quality=92)
    shutil.copy2(DRIVE_PAGES / name, REPO_PAGES / name)
    print(f"  {name}  {img.size[0]}x{img.size[1]}  -> Drive and repo")

print(f"\nDrive copy: {DRIVE_PAGES}")
print("Download that folder into paper/figures/pages/ locally, then run")
print("    python paper/build_four_quadrant.py")

In [ ]:
# Rebuild the figure here so the result is visible before leaving Colab. The
# builder asserts each panel's entropy and verdict against the public manifest,
# so a mislabelled panel fails loudly rather than rendering quietly.
import subprocess

from IPython.display import Image, display

r = subprocess.run([sys.executable, "paper/build_four_quadrant.py"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout or "", r.stderr or "")
r.check_returncode()
display(Image(filename=f"{REPO_DIR}/paper/figures/four_quadrant.png"))